# Tutorial 04 — TNO orbit fit from real MPC astrometry

Given the recorded MPC astrometry (RA, Dec, JD) for a known Trans-Neptunian Object,
Ariadne builds nightly tracklets, re-derives the (r, ṙ) hypothesis on the candidate's
own data, and refines a full 6D heliocentric state via LM differential correction.

This is the discrimination filter at the heart of the discovery engine: real orbits fit
to single-digit arcseconds; mixed-object false-positive clusters fit to hundreds of
arcsec or fail outright.

_~1 minute to run (network fetch + IOD search per target)._

In [ ]:
import math, warnings
warnings.filterwarnings('ignore')
import numpy as np
import ariadne
from ariadne.data.constants import GM_SUN, AU_KM

In [ ]:
# Real TNOs from MPC astrometry, JPL reference elements for comparison
TARGETS = [
    ('Sedna',  '90377',  {'a': 506.0, 'e': 0.85, 'i': 11.93}),
    ('Eris',   '136199', {'a':  67.7, 'e': 0.44, 'i': 44.04}),
    ('Quaoar', '50000',  {'a':  43.2, 'e': 0.04, 'i':  7.99}),
]
for label, desig, jpl in TARGETS:
    print(f'\n--- {label} (JPL: a={jpl["a"]} AU, e={jpl["e"]}, i={jpl["i"]} deg)')
    fit = ariadne.discover_tno(desig, window_days=720)
    r, v = np.asarray(fit['x_fit']), np.asarray(fit['v_fit'])
    rn, vn = float(np.linalg.norm(r)), float(np.linalg.norm(v))
    a_au = (1.0 / (2.0 / rn - vn ** 2 / GM_SUN)) / AU_KM
    h = np.cross(r, v); hn = float(np.linalg.norm(h))
    ecc = float(np.linalg.norm(np.cross(v, h) / GM_SUN - r / rn))
    inc = math.degrees(math.acos(max(-1, min(1, h[2] / hn))))
    grade = 'EXCELLENT' if fit['rms_arcsec'] < 1 else 'GOOD' if fit['rms_arcsec'] < 10 else 'POOR'
    print(f'  IOD seed: r={fit["iod"]["r_au"]:.1f} AU, rdot={fit["iod"]["rdot"]:+.2f} km/s')
    print(f'  FIT: a={a_au:.1f} AU, e={ecc:.3f}, i={inc:.2f} deg, RMS={fit["rms_arcsec"]:.2f}"  [{grade}]')

## What just happened

`ariadne.discover_tno(designation)` fetched real MPC astrometry, built nightly tracklets,
ran the IOD hypothesis-search (a fine grid of (r, ṙ) hypotheses, picking the one that
makes the cluster scatter smallest at the reference epoch), then LM-refined a full 6D
heliocentric state with light-time correction.

The discovery filter accepts orbits with RMS < ~10″. For real coherent observations the
fit comes out to a few arcseconds (Sedna 3.94″, Quaoar 3.79″, Eris 5.79″) and the
semi-major axis recovery is within 0.1–13%. For mixed-object clusters the fit fails or
lands at hundreds of arcseconds — the sharp two-state behavior a discovery filter needs.